# EJ Screen Climate Change Indicators

This notebook is intended to respond to the question "Which communities are most at risk for the impacts of climate change?" using EJ Screen (2024) climate change indicators and demographic index.

#1. Import Dependencies
Import the required libraries and mount your google drive (use one with permissions for the EDGI shared drive)

In [ ]:
import pandas as pd
import geopandas as gpd

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#2. Load EJ Screen

In [ ]:
# Load EJScreen/EJAM data
ejscreen = gpd.read_file("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb").drop(columns="geometry")

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb': 'EJSCREEN_Full_with_AS_CNMI_GU_VI' (default), 'USA'. Specify layer parameter to avoid this warning.
  result = read_func(


In [ ]:
ejscreen_columns = ejscreen.columns.tolist()
ejscreen_columns

['ID',
 'STATE_NAME',
 'ST_ABBREV',
 'CNTY_NAME',
 'REGION',
 'ACSTOTPOP',
 'ACSIPOVBAS',
 'ACSEDUCBAS',
 'ACSTOTHH',
 'ACSTOTHU',
 'ACSUNEMPBAS',
 'ACSDISABBAS',
 'DEMOGIDX_2',
 'DEMOGIDX_5',
 'PEOPCOLOR',
 'PEOPCOLORPCT',
 'LOWINCOME',
 'LOWINCPCT',
 'UNEMPLOYED',
 'UNEMPPCT',
 'DISABILITY',
 'DISABILITYPCT',
 'LINGISO',
 'LINGISOPCT',
 'LESSHS',
 'LESSHSPCT',
 'UNDER5',
 'UNDER5PCT',
 'OVER64',
 'OVER64PCT',
 'LIFEEXPPCT',
 'PM25',
 'OZONE',
 'DSLPM',
 'RSEI_AIR',
 'PTRAF',
 'PRE1960',
 'PRE1960PCT',
 'PNPL',
 'PRMP',
 'PTSDF',
 'UST',
 'PWDIS',
 'NO2',
 'DWATER',
 'D2_PM25',
 'D5_PM25',
 'D2_OZONE',
 'D5_OZONE',
 'D2_DSLPM',
 'D5_DSLPM',
 'D2_RSEI_AIR',
 'D5_RSEI_AIR',
 'D2_PTRAF',
 'D5_PTRAF',
 'D2_LDPNT',
 'D5_LDPNT',
 'D2_PNPL',
 'D5_PNPL',
 'D2_PRMP',
 'D5_PRMP',
 'D2_PTSDF',
 'D5_PTSDF',
 'D2_UST',
 'D5_UST',
 'D2_PWDIS',
 'D5_PWDIS',
 'D2_NO2',
 'D5_NO2',
 'D2_DWATER',
 'D5_DWATER',
 'P_DEMOGIDX_2',
 'P_DEMOGIDX_5',
 'P_PEOPCOLORPCT',
 'P_LOWINCPCT',
 'P_UNEMPPCT',
 'P_DISABI

#3. Load EJ Screen Climate Change Indicators

Load data for EJ Screen Climate Change indicators from EDGI's Google Drive. These are not yet part of the EJ Screen file above, so we'll need to prepare them and then merge them.

In [ ]:
flood_risk = gpd.read_file("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/EPA_EJScreen_Flood_Risk.gdb")

,ID,ST_ABBREV,T_flood_y00,T_flood_y30,PERCENTILE,SHAPE_Length,SHAPE_Area,geometry
0,010010201001,AL,95 %ile,94 %ile,95.0,0.110793,0.000412,"MULTIPOLYGON (((-86.51038 32.47225, -86.5103 3..."
1,010010201002,AL,45 %ile,42 %ile,45.0,0.096868,0.000534,"MULTIPOLYGON (((-86.50461 32.47723, -86.50453 ..."
2,010010202001,AL,40 %ile,40 %ile,40.0,0.062129,0.000197,"MULTIPOLYGON (((-86.48127 32.47744, -86.48126 ..."
3,010010202002,AL,94 %ile,93 %ile,94.0,0.052097,0.000122,"MULTIPOLYGON (((-86.47611 32.46765, -86.47564 ..."
4,010010203001,AL,42 %ile,42 %ile,42.0,0.089441,0.000372,"MULTIPOLYGON (((-86.47087 32.47573, -86.47084 ..."
...,...,...,...,...,...,...,...,...
242743,None,None,None,None,NaN,0.087447,0.000248,"MULTIPOLYGON (((-64.92651 18.3303, -64.9264 18..."
242744,None,None,None,None,NaN,0.046377,0.000049,"MULTIPOLYGON (((-64.92367 18.34292, -64.92348 ..."
242745,None,None,None,None,NaN,0.029631,0.000050,"MULTIPOLYGON (((-64.92958 18.34448, -64.92933 ..."
242746,None,None,None,None,NaN,0.055502,0.000113,"MULTIPOLYGON (((-64.93138 18.33971, -64.93076 ..."


In [ ]:
state_abbrev_to_name = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico",
    "NY": "New York", "NC": "North Carolina", "ND": "North Dakota",
    "OH": "Ohio", "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania",
    "RI": "Rhode Island", "SC": "South Carolina", "SD": "South Dakota",
    "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
    "DC": "District of Columbia", "PR": "Puerto Rico"
}


In [ ]:
flood_risk["STATE_NAME"] = flood_risk["ST_ABBREV"].map(
    state_abbrev_to_name
)


In [ ]:
heat_risk = gpd.read_file("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/Heat_Index.gdb")
heat_risk

,ID,STATE_NAME,Days_Above_90_2019,Days_Above_90_2020,Days_Above_90_2021,Days_Above_90_2022,Days_Above_90_2023,Max_Days_Above_90,Average_Days_Above_90,SHAPE_Length,SHAPE_Area,geometry
0,7801071600,Virgin Islands,34.0,100.0,41.0,55.0,115.0,115.0,69.0,4682.858860,1.345750e+06,"MULTIPOLYGON (((-7208355.462 2007572.482, -720..."
1,7801064915,Virgin Islands,34.0,100.0,41.0,55.0,115.0,115.0,69.0,6041.659016,2.006416e+06,"MULTIPOLYGON (((-7223273.101 2005923.808, -722..."
2,7801008070,Virgin Islands,34.0,100.0,41.0,55.0,115.0,115.0,69.0,3346.524431,6.694057e+05,"MULTIPOLYGON (((-7212124.964 2010579.425, -721..."
3,7801024500,Virgin Islands,34.0,100.0,41.0,55.0,115.0,115.0,69.0,4629.168276,1.307798e+06,"MULTIPOLYGON (((-7221659.926 2003250.838, -722..."
4,7803026350,Virgin Islands,34.0,100.0,41.0,55.0,115.0,115.0,69.0,566.389287,1.604417e+04,"MULTIPOLYGON (((-7230296.096 2077420.877, -723..."
...,...,...,...,...,...,...,...,...,...,...,...,...
243015,721537506011,Puerto Rico,46.0,58.0,50.0,32.0,96.0,96.0,57.0,24923.329900,1.047352e+07,"MULTIPOLYGON (((-7443526.754 2039806.619, -744..."
243016,721537506012,Puerto Rico,46.0,58.0,50.0,32.0,96.0,96.0,57.0,2183.057961,2.957322e+05,"MULTIPOLYGON (((-7441399.549 2040143.289, -744..."
243017,721537506013,Puerto Rico,46.0,58.0,50.0,32.0,96.0,96.0,57.0,6723.139329,1.449218e+06,"MULTIPOLYGON (((-7442760.764 2039059.9, -74427..."
243018,721537506021,Puerto Rico,46.0,58.0,50.0,32.0,96.0,96.0,57.0,5128.438096,1.265276e+06,"MULTIPOLYGON (((-7444594.976 2039281.958, -744..."


In [ ]:
heat_risk.columns

Index(['ID', 'STATE_NAME', 'Days_Above_90_2019', 'Days_Above_90_2020',
       'Days_Above_90_2021', 'Days_Above_90_2022', 'Days_Above_90_2023',
       'Max_Days_Above_90', 'Average_Days_Above_90', 'SHAPE_Length',
       'SHAPE_Area', 'geometry'],
      dtype='object')

In [ ]:
wildfire_risk = gpd.read_file("/content/drive/Shareddrives/EDGI - Shared NEW/12_Environmental_Enforcement_Watch/09_Data/Wildfire_Risk.gdb")

In [ ]:
wildfire_risk["STATE_NAME"] = wildfire_risk["ST_ABBREV"].map(
    state_abbrev_to_name
)

In [ ]:
contiguous_states = [
    "Alabama", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts",
    "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska",
    "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming"
]


In [ ]:
flood_risk = flood_risk[flood_risk["STATE_NAME"].isin(contiguous_states)]
heat_risk = heat_risk[heat_risk["STATE_NAME"].isin(contiguous_states)]
wildfire_risk = wildfire_risk[wildfire_risk["STATE_NAME"].isin(contiguous_states)]


In [ ]:
ejscreen["ID"] = ejscreen["ID"].astype(str).str.zfill(12)
wildfire_risk["ID"] = wildfire_risk["ID"].astype(str).str.zfill(12)
flood_risk["ID"] = flood_risk["ID"].astype(str).str.zfill(12)
heat_risk["ID"] = heat_risk["ID"].astype(str).str.zfill(12)


/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


#4. Calculate Heat Risk percentiles for consistency

It might be helpful to also see heat risk as a percentile for ease of comparing with the other two variables. We'll calculate that percentile based on the maximum number of days above 90 degress F for each census block group, finding their national percentile rankings.

In [ ]:
heat_risk["Max_Days_Above_90"] = pd.to_numeric(
    heat_risk["Max_Days_Above_90"],
  #  errors="coerce"
)


In [ ]:
heat_risk["PERCENTILES"] = (
    heat_risk["Max_Days_Above_90"]
    .rank(pct=True)
    * 100
)


/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
heat_risk["PERCENTILES"] = heat_risk["PERCENTILES"].round(2)


In [ ]:
heat_risk[["Max_Days_Above_90","PERCENTILES"]].describe()

,Max_Days_Above_90,PERCENTILES
count,237622.000000,237622.000000
mean,53.666041,50.000027
std,45.670607,28.864636
min,0.000000,1.520000
25%,16.000000,24.550000
50%,37.000000,50.070000
75%,87.000000,74.900000
max,222.000000,100.000000


Here we choose some specific variables we want to keep from the climate change indicators.

In [ ]:
# wildfire = wildfire_risk[["ID", "STATE_NAME", "PERCENTILE", "T_fire_y30"]]
# flood = flood_risk[["ID", "STATE_NAME", "PERCENTILE", "T_flood_y30"]]
# heat = heat_risk[["ID", "STATE_NAME", "PERCENTILES"]]


And make sure to rename "PERCENTILE" columns from the different data sources so it's less confusing after the join.

#5. Then merge them with EJ Screen based on their census block group IDs

In [ ]:
ej_cc = (
    ejscreen
    .rename(columns={
        "P_PM25": "pm25_pct",
        "P_NO2": "no2_pct",
        "P_OZONE": "ozone_pct",
        "P_RSEI_AIR": "rsei_pct",
        "P_DSLPM": "dieselpm_pct",
        "P_PTRAF": "traffic_pct"
    })
    .merge(
        wildfire_risk[["ID", "PERCENTILE", "T_fire_y30"]].rename(columns={"PERCENTILE": "wildfire_pct"}),
        on="ID", how="left"
    )
    .merge(
        flood_risk[["ID", "PERCENTILE", "T_flood_y30"]].rename(columns={"PERCENTILE": "flood_pct"}),
        on="ID", how="left"
    )
    .merge(
        heat_risk[["ID", "PERCENTILES"]].rename(columns={"PERCENTILES": "heat_pct"}),
        on="ID", how="left"
    )
)

In [ ]:
ej_cc = ej_cc.loc[
    (ej_cc["pm25_pct"].notna()) &
    (ej_cc["dieselpm_pct"].notna()) &
    (ej_cc["rsei_pct"].notna()) &
    (ej_cc["ozone_pct"].notna()) &
    (ej_cc["no2_pct"].notna()) &
    (ej_cc["traffic_pct"].notna()) &
    (ej_cc["wildfire_pct"].notna()) &
    (ej_cc["heat_pct"].notna()) &
    (ej_cc["flood_pct"].notna())
].copy()


Thresholds (80th %ile)

Here we will use a threshold to identify block groups that are greater than or equal to the 80th %ile for flood (2030), wildfire (2030), and extreme heat. Later, we'll calculate the % of block groups within a congressional district that meet these criteria.

In [ ]:
#Convert %iles from string to number for 2030 model

In [ ]:
ej_cc["flood_pct_2030"] = (
    ej_cc["T_flood_y30"]
    .str.replace("%ile", "", regex=False)
    .astype(int)
)
ej_cc["wildfire_pct_2030"] = (
    ej_cc["T_fire_y30"]
    .str.replace("%ile", "", regex=False)
    .astype(int)
)

In [ ]:
#Air Quality + demographic threshold
ej_cc["airquality_demog_high"] = (
    (ej_cc["pm25_pct"] >= 80)|
    (ej_cc["dieselpm_pct"] >= 80)|
    (ej_cc["rsei_pct"] >= 80)|
    (ej_cc["ozone_pct"] >= 80)|
    (ej_cc["no2_pct"] >= 80)|
    (ej_cc["traffic_pct"] >= 80) &
    (
    (ej_cc["P_DEMOGIDX_2"] >= 80)|
    (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)

# Extreme heat risk + demographic threshold
ej_cc["heat_demog_high"] = (
    (ej_cc["heat_pct"] >= 80) &
    (
    (ej_cc["P_DEMOGIDX_2"] >= 80)|
    (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)

# Flood risk + demographic threshold for 2020 data
ej_cc["flood_demog_high"] = (
    (ej_cc["flood_pct"] >= 80) &
    (
        (ej_cc["P_DEMOGIDX_2"] >= 80) |
        (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)

# Wildfire risk + demographic threshold for 2020 data
ej_cc["wildfire_demog_high"] = (
    (ej_cc["wildfire_pct"] >= 80) &
    (
    (ej_cc["P_DEMOGIDX_2"] >= 80)|
    (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)


In [ ]:
# Flood risk for 2030 model
ej_cc["flood_demog_high_2030"] = (
    (ej_cc["flood_pct_2030"] >= 80) &
    (
        (ej_cc["P_DEMOGIDX_2"] >= 80) |
        (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)


# Wildfire risk for 2030 model
ej_cc["wildfire_demog_high_2030"] = (
    (ej_cc["wildfire_pct_2030"] >= 80) &
    (
    (ej_cc["P_DEMOGIDX_2"] >= 80)|
    (ej_cc["P_DEMOGIDX_5"] >= 80)
    )
).astype(int)


In [ ]:
ej_cc_columns = ej_cc.columns.tolist()
ej_cc_columns

['ID',
 'STATE_NAME',
 'ST_ABBREV',
 'CNTY_NAME',
 'REGION',
 'ACSTOTPOP',
 'ACSIPOVBAS',
 'ACSEDUCBAS',
 'ACSTOTHH',
 'ACSTOTHU',
 'ACSUNEMPBAS',
 'ACSDISABBAS',
 'DEMOGIDX_2',
 'DEMOGIDX_5',
 'PEOPCOLOR',
 'PEOPCOLORPCT',
 'LOWINCOME',
 'LOWINCPCT',
 'UNEMPLOYED',
 'UNEMPPCT',
 'DISABILITY',
 'DISABILITYPCT',
 'LINGISO',
 'LINGISOPCT',
 'LESSHS',
 'LESSHSPCT',
 'UNDER5',
 'UNDER5PCT',
 'OVER64',
 'OVER64PCT',
 'LIFEEXPPCT',
 'PM25',
 'OZONE',
 'DSLPM',
 'RSEI_AIR',
 'PTRAF',
 'PRE1960',
 'PRE1960PCT',
 'PNPL',
 'PRMP',
 'PTSDF',
 'UST',
 'PWDIS',
 'NO2',
 'DWATER',
 'D2_PM25',
 'D5_PM25',
 'D2_OZONE',
 'D5_OZONE',
 'D2_DSLPM',
 'D5_DSLPM',
 'D2_RSEI_AIR',
 'D5_RSEI_AIR',
 'D2_PTRAF',
 'D5_PTRAF',
 'D2_LDPNT',
 'D5_LDPNT',
 'D2_PNPL',
 'D5_PNPL',
 'D2_PRMP',
 'D5_PRMP',
 'D2_PTSDF',
 'D5_PTSDF',
 'D2_UST',
 'D5_UST',
 'D2_PWDIS',
 'D5_PWDIS',
 'D2_NO2',
 'D5_NO2',
 'D2_DWATER',
 'D5_DWATER',
 'P_DEMOGIDX_2',
 'P_DEMOGIDX_5',
 'P_PEOPCOLORPCT',
 'P_LOWINCPCT',
 'P_UNEMPPCT',
 'P_DISABI

In [ ]:
ej_cc.to_csv("ej_cc_threshold_contiguous_us.csv", index = False)

In [ ]:
ej_cc.to_parquet("ej_cc_threshold_contiguous_us.parquet")